# Lesson 9: Linear regression on daily moves of the US Treasury curve

1. What does a line say about the 10-year yield's daily move?
2. Which line is best, and how is it found?
3. How good is the line on days it has not seen?
4. When does a model learn too little or too much?

A move is one day's yield minus the yield of the trading day before, in basis points (bp); 1 bp is 0.01 percentage points. The 5,164 daily moves run from 10 February 2006 to 30 September 2026. Lines are fit on the 4,131 days to 11 August 2022 and tested on the 1,033 days from 12 August 2022.

Treasury yields: Federal Reserve Board H.15 via FRED, Federal Reserve Bank of St. Louis; snapshot 30 September 2026.

In [ ]:
import pathlib

import pandas as pd

DATA_URL = "https://digital-ai-finance.github.io/data-science-msc/data/"
SNAPSHOT = "2026-09-30"


def load(name):
    """Read a snapshot: "treasury-curve-daily"."""
    local = pathlib.Path("../data/market") / f"{SNAPSHOT}-{name}.csv"
    if local.exists():  # run from material/notebooks/ before the files are published
        return pd.read_csv(local)
    url = f"{DATA_URL}{SNAPSHOT}-{name}.csv"
    try:
        return pd.read_csv(url)
    except Exception as err:
        raise RuntimeError(f"{url} not reachable ({err}). Download the file in a browser, upload "
                           "it in the Colab file panel and read it with pd.read_csv.") from err

In [ ]:
curve = load("treasury-curve-daily")
curve["date"] = pd.to_datetime(curve["date"])
curve = curve.set_index("date")
moves = curve.diff().dropna().mul(100).round(2)     # daily moves in basis points
train = moves.loc[:"2022-08-11"]                    # split by date: the test days come last
test = moves.loc["2022-08-12":]
print(curve.shape, moves.shape, train.shape, test.shape)
moves.head()

## 1. What does a line say about the 10-year yield's daily move?

The line $\hat{y} = \beta_0 + \beta_1 x$ predicts the 10-year move $\hat{y}$ from the 2-year move $x$. The intercept $\beta_0$ is the predicted move on a day the 2-year yield does not move; the slope $\beta_1$ is the change of the prediction per basis point of the 2-year move.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x, y = moves["DGS2"], moves["DGS10"]
slope, intercept = np.polyfit(x, y, 1)
plt.scatter(x, y, s=2, alpha=0.3)
grid = np.linspace(x.min(), x.max(), 2)
plt.plot(grid, intercept + slope * grid, color="tab:orange")
plt.xlabel("2-year move (bp)")
plt.ylabel("10-year move (bp)")
plt.show()
print(f"{len(moves):,} moves: intercept {intercept:.2f} bp, slope {slope:.3f}")
big = y.abs().idxmax()
print(f"largest 10-year move: {y[big]:.0f} bp on {big.date()}, 2-year move {x[big]:.0f} bp")

In [ ]:
from sklearn.linear_model import LinearRegression

print(f"slope times 10: {10 * slope:.2f} bp")
print(f"predicted 10-year move for a 2-year move of +10 bp: {intercept + slope * 10:.2f} bp")
two = LinearRegression().fit(moves[["DGS2", "DGS30"]], moves["DGS10"])
print(f"2-year and 30-year moves: intercept {two.intercept_:.4f} bp, "
      f"coefficients {two.coef_[0]:.2f} and {two.coef_[1]:.2f}")

**Exercise 1.** Predict the 10-year move for a day with a 2-year move of +15 bp and a 30-year move of +5 bp, once with the 2-year line and once with the model on both moves.

## 2. Which line is best, and how is it found?

A residual is the observed move minus the predicted one, $e = y - \hat{y}$. Least squares picks the line with the smallest sum of squared residuals. The normal equation $X^\top X \beta = X^\top y$ gives that line in one step; gradient descent reaches it in many small steps, each of a size the learning rate sets.

In [ ]:
x6 = np.array([-12, -8, -4, 4, 8, 12], dtype=float)     # 2-year move, bp, stylised
y6 = np.array([-9, -4, 0, 0, 8, 11], dtype=float)       # 10-year move, bp, stylised
lines = {"A: 2 + 0.75x": (2, 0.75), "B: 1 + 0.75x": (1, 0.75), "C: x": (0, 1)}
for name, (b0, b1) in lines.items():
    e = y6 - (b0 + b1 * x6)
    print(f"{name:13s} residuals {e}  sum of squares {np.sum(e**2):.0f}")

In [ ]:
def fit_normal_equation(X, y):
    """Fit linear regression using the normal equation."""
    return np.linalg.solve(X.T @ X, X.T @ y)


X6 = np.column_stack([np.ones_like(x6), x6])
b = fit_normal_equation(X6, y6)
print(f"six days: intercept {b[0]:.2f} bp, slope {b[1]:.2f}")
Xtr = np.column_stack([np.ones(len(train)), train["DGS2"]])
b = fit_normal_equation(Xtr, train["DGS10"].to_numpy())
print(f"training days: intercept {b[0]:.2f} bp, slope {b[1]:.3f}")

In [ ]:
from scipy import stats

e6 = y6 - X6 @ fit_normal_equation(X6, y6)  # residuals of the least-squares line on the six days
s2 = e6 @ e6 / (len(x6) - 2)
se = np.sqrt(s2 * np.linalg.inv(X6.T @ X6)[1, 1])
t = stats.t.ppf(0.975, len(x6) - 2)
print(f"six days: s^2 {s2:.1f}, standard error of the slope {se:.3f}, t {t:.3f}, "
      f"95 per cent interval {0.75 - t * se:.2f} to {0.75 + t * se:.2f}")

In [ ]:
def fit_gradient_descent(X, y, learning_rate, n_iterations=30):
    """Gradient descent on the mean squared error; the coefficients and the loss per step."""
    beta, losses = np.zeros(X.shape[1]), []
    for _ in range(n_iterations):
        residual = y - X @ beta
        losses.append(np.mean(residual**2))
        beta = beta + learning_rate * 2 * X.T @ residual / len(y)
    return beta, losses


z = (train["DGS2"] - train["DGS2"].mean()) / train["DGS2"].std(ddof=0)
Xz = np.column_stack([np.ones(len(z)), z])
for rate in (0.01, 0.1, 1.1):
    beta, losses = fit_gradient_descent(Xz, train["DGS10"].to_numpy(), rate)
    plt.plot(losses, label=f"learning rate {rate}")
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("mean squared error (bp²)")
plt.legend()
plt.show()

`fit_gradient_descent` divides the squared error by the number of days and runs on the standardised 2-year move, which makes its error curve 448 times flatter than the sum of squared residuals of the six days in raw bp. A learning rate 448 times larger takes the same step: 0.01 and 0.1 close in on the minimum, and 1.1, about 448 × 0.0025, overshoots it as α = 0.0025 does on the six days.

**Exercise 2.** Run `fit_gradient_descent` with the learning rates 0.5, 0.9, 1.0 and 1.05. Which is the largest rate that still lowers the loss?

## 3. How good is the line on days it has not seen?

RMSE is the square root of the mean squared residual and MAE the mean absolute residual, both in bp. $R^2 = 1 - \sum (y - \hat{y})^2 / \sum (y - \bar{y})^2$ is the share of the variation in the 10-year move that a line explains. The training mean, the average move of the training days, predicts the same move for every test day and sets the error any model has to beat.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rows = []
for name, cols in {"2-year": ["DGS2"], "2-year and 30-year": ["DGS2", "DGS30"]}.items():
    model = LinearRegression().fit(train[cols], train["DGS10"])
    for part, df in {"train": train, "test": test}.items():
        pred = model.predict(df[cols])
        rows.append({"model": name, "days": part,
                     "R2": r2_score(df["DGS10"], pred),
                     "RMSE": mean_squared_error(df["DGS10"], pred) ** 0.5,
                     "MAE": mean_absolute_error(df["DGS10"], pred)})
rows.append({"model": "training mean", "days": "test",
             "RMSE": ((test["DGS10"] - train["DGS10"].mean()) ** 2).mean() ** 0.5})


def fmt(digits):
    return lambda v: "" if pd.isna(v) else f"{v:.{digits}f}"


table = pd.DataFrame(rows)
print(table.to_string(index=False, na_rep="", formatters={"R2": fmt(3), "RMSE": fmt(2), "MAE": fmt(2)}))

In [ ]:
simple = LinearRegression().fit(train[["DGS2"]], train["DGS10"])
fitted = simple.predict(test[["DGS2"]])
resid = test["DGS10"] - fitted
plt.scatter(fitted, resid, s=4, alpha=0.4)
plt.axhline(0, color="black")
plt.xlabel("predicted 10-year move (bp)")
plt.ylabel("residual (bp)")
plt.show()
size = test["DGS2"].abs()
for label, mask in [("at most 4 bp", size <= 4), ("10 bp or more", size >= 10)]:
    print(f"2-year move {label}: {int(mask.sum())} test days, residual sd {resid[mask].std():.2f} bp")

**Exercise 3.** Write `rmse(y_true, y_pred)` with NumPy and compare its value on the test days with `mean_squared_error(y_true, y_pred) ** 0.5`.

## 4. When does a model learn too little or too much?

Ten training days with ten different 2-year moves are fit by polynomials of degree 1 to 9, and each polynomial is tested on the test days inside the range of the ten. Ten points fix a polynomial of degree 9 exactly. The learning curve fits the 2-year line on 20 to 4,131 training days, 20 random draws per size.

In [ ]:
rng = np.random.default_rng(42)
pool = train.drop_duplicates("DGS2")                # one training day per distinct 2-year move
sub = pool.iloc[rng.choice(len(pool), 10, replace=False)]
m, s = sub["DGS2"].mean(), sub["DGS2"].std()
inside = test[(test["DGS2"] >= sub["DGS2"].min()) & (test["DGS2"] <= sub["DGS2"].max())]
z, zt = (sub["DGS2"] - m) / s, (inside["DGS2"] - m) / s
result = []
for d in range(1, 10):
    c = np.polyfit(z, sub["DGS10"], d)
    result.append((d, np.sqrt(np.mean((sub["DGS10"] - np.polyval(c, z)) ** 2)),
                   np.sqrt(np.mean((inside["DGS10"] - np.polyval(c, zt)) ** 2))))
degrees = pd.DataFrame(result, columns=["degree", "train RMSE", "test RMSE"])
degrees.plot(x="degree", logy=True, marker="o")
plt.ylabel("RMSE (bp)")
plt.show()
print(f"{len(inside):,} test days inside the range of the 10 training days")
for d, tr, te in result:
    print(f"degree {d}: train RMSE {tr:.2f} bp, test RMSE {te:.2f} bp")
print("lowest test RMSE at degree", degrees.loc[degrees["test RMSE"].idxmin(), "degree"])

In [ ]:
xa, ya = train["DGS2"].to_numpy(), train["DGS10"].to_numpy()
xt, yt = test["DGS2"].to_numpy(), test["DGS10"].to_numpy()
rows = []
for n in (20, 50, 100, 200, 500, 1000, 2000, 4131):
    tr_err, te_err = [], []
    for _ in range(20):
        k = rng.choice(len(xa), n, replace=False)
        c = np.polyfit(xa[k], ya[k], 1)
        tr_err.append(np.sqrt(np.mean((ya[k] - np.polyval(c, xa[k])) ** 2)))
        te_err.append(np.sqrt(np.mean((yt - np.polyval(c, xt)) ** 2)))
    rows.append((n, np.mean(tr_err), np.mean(te_err)))
lc = pd.DataFrame(rows, columns=["training days", "train RMSE", "test RMSE"])
lc.plot(x="training days", logx=True, marker="o")
plt.ylabel("RMSE (bp)")
plt.show()
for n, tr, te in rows:
    print(f"{n:5,d} training days: train RMSE {tr:.2f}, test RMSE {te:.2f}, gap {te - tr:.2f} bp")

**Exercise 4.** Fit the 10-year move on the 1-month move `DGS1MO` alone and compare its training and test RMSE with the test RMSE of the training mean. Then add the 2-year and 30-year moves.

## Least squares picks the line, and on test days it misses by 3.92 bp

1. Over the 5,164 days the 10-year move rises 0.808 bp per basis point of the 2-year move; beside the 30-year move, the 2-year coefficient falls to 0.34.
2. Least squares picks the line with the smallest sum of squared residuals, 24 for the six days against 30 for line A; gradient descent reaches it in steps whose size the learning rate sets.
3. On the 1,033 test days from 12 August 2022 the 2-year line misses the 10-year move by 3.92 bp (RMSE), against 6.10 bp for the training mean, and the residuals spread wider on days of large moves.
4. Training and test error both high and close mean underfitting; training error low and test error high mean overfitting, as the degree-9 polynomial through 10 days shows.

## 5. Newer data (optional)

Live 2-year and 10-year yields from FRED, from 9 February 2006 to the latest day, refit with the same line. Their numbers differ from the slides, which use the snapshot of 30 September 2026.

In [ ]:
FRED = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="
BIS = "https://stats.bis.org/api/v1/data/{}?startPeriod={}&format=csv"


def fred(*ids):
    """Live FRED series, e.g. fred("GS10", "TB3MS", "USREC"); "." marks a missing value."""
    return pd.read_csv(FRED + ",".join(ids), na_values=["."], parse_dates=["observation_date"])


def bis(flow, start):
    """Live BIS series, e.g. bis("WS_CBPOL/M..", "2021-12")."""
    return pd.read_csv(BIS.format(flow, start))


live = fred("DGS2", "DGS10").set_index("observation_date").dropna()
live_moves = live.loc["2006-02-09":].diff().dropna().mul(100)
b1, b0 = np.polyfit(live_moves["DGS2"], live_moves["DGS10"], 1)
print(f"{len(live_moves):,} moves to {live_moves.index[-1].date()}: "
      f"slope {b1:.3f}, intercept {b0:.3f} bp")